# Structured Output (JSON Schema)

This notebook demonstrates three ways to get structured JSON from GigaChat:

1. **Raw dict schema** + `json.loads` — pass a JSON Schema dict as-is (no normalization).
2. **Pydantic model as schema** — SDK auto-generates and normalizes the JSON Schema.
3. **`client.chat.legacy.parse()` helper** — one-step send, parse, and validate.

Import GigaChat client and required models.

In [ ]:
import os
import getpass
import json
from typing import List

from pydantic import BaseModel

from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

Set up GigaChat credentials and scope. The input will be hidden for security.

In [ ]:
if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("GigaChat Credentials: ")

if "GIGACHAT_SCOPE" not in os.environ:
    os.environ["GIGACHAT_SCOPE"] = getpass.getpass("GigaChat Scope: ")

## Define the output schema

A Pydantic model describing the desired response shape. The same model is reused across all three approaches.

In [7]:
class MathAnswer(BaseModel):
    steps: List[str]
    final_answer: str


PROMPT = "Solve the equation 8x + 7 = -23. Explain step by step in English."

## 1. Raw dict schema

Pass a plain JSON Schema dictionary via `response_format`. The schema is sent as-is with no normalization. You parse the JSON yourself.

In [8]:
chat = Chat(
    messages=[Messages(role=MessagesRole.USER, content=PROMPT)],
    response_format={
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "steps": {"type": "array", "items": {"type": "string"}},
                "final_answer": {"type": "string"},
            },
            "required": ["steps", "final_answer"],
        },
        "strict": False,
    },
)

with GigaChat(verify_ssl_certs=False) as client:
    resp = client.chat.legacy.create(chat)

data = json.loads(resp.choices[0].message.content)
print("=== Raw dict schema ===")
print("Steps:", data["steps"])
print("Answer:", data["final_answer"])

=== Raw dict schema ===
Steps: ['Solve for x by isolating x on one side of the equation.', '', 'Step 1: Subtract 7 from both sides to isolate the term with x.', '8x + 7 - 7 = -23 - 7', '8x = -30', '', 'Step 2: Divide both sides by 8 to solve for x.', '8x / 8 = -30 / 8', 'x = -30 / 8', '', 'Step 3: Simplify the fraction if possible.', '-30 / 8 can be simplified to -15 / 4.']
Answer: x = -\frac{15}{4}


## 2. Pydantic model as schema

Pass a Pydantic `BaseModel` class — the SDK generates and normalizes the JSON Schema automatically.

In [9]:
chat = Chat(
    messages=[Messages(role=MessagesRole.USER, content=PROMPT)],
    response_format={
        "type": "json_schema",
        "schema": MathAnswer,
        "strict": True,
    },
)

with GigaChat(verify_ssl_certs=False) as client:
    resp = client.chat.legacy.create(chat)

data = json.loads(resp.choices[0].message.content)
parsed = MathAnswer.model_validate(data)
print("=== Pydantic model as schema ===")
print("Steps:", parsed.steps)
print("Answer:", parsed.final_answer)

=== Pydantic model as schema ===
Steps: ['Start with the equation $8x + 7 = -23$.', 'Subtract $7$ from both sides to isolate the term containing $x$:\n       $8x + 7 - 7 = -23 - 7$ which simplifies to $8x = -30$.', 'Divide both sides by $8$ to solve for $x$:\n       $x = \\frac{-30}{8}$.', 'Simplify the fraction $\\frac{-30}{8}$ by dividing numerator and denominator by $2$:\n       $x = \\frac{-15}{4}$.', 'Convert $\\frac{-15}{4}$ to a decimal if desired: $-\\frac{15}{4} = -3.75$.']
Answer: -\dfrac{15}{4}


## 3. `client.chat.legacy.parse()` helper

One call to send the request, parse JSON, and validate against the Pydantic model. Returns both the raw `ChatCompletion` and the parsed object.

In [ ]:
with GigaChat(verify_ssl_certs=False) as client:
    completion, parsed = client.chat.legacy.parse(
        PROMPT,
        response_format=MathAnswer,
        strict=True,
    )

print("=== client.chat.legacy.parse() helper ===")
print("Steps:", parsed.steps)
print("Answer:", parsed.final_answer)
print(f"Tokens used: {completion.usage.total_tokens}")